In [ ]:
import os
import re
import glob
import torch
import tempfile
import numpy as np
from datasets import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders, normalizers
from transformers import (
    AutoTokenizer, PreTrainedTokenizerFast,
    LlamaConfig, LlamaForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    TrainerCallback
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Загрузка и препроцессинг данных

In [ ]:
def load_and_preprocess(data_dir="data"):
    print("Загрузка текстов...")
    texts = []
    for fpath in glob.glob(os.path.join(data_dir, "**/*.txt"), recursive=True):
        with open(fpath, "r", encoding="utf-8") as f:
            texts.append(f.read())

    raw_text = "\n".join(texts)
    lines = raw_text.split("\n")

    print("🧹 Очистка и фильтрация...")
    latin_re = re.compile(r'[A-Za-z]')
    cleaned = []

    for line in lines:
        line = line.strip()
        if not line: continue

        # Удаляем строки с латинскими буквами
        if latin_re.search(line): continue

        # Нормализация повторяющейся пунктуации
        line = re.sub(r'([.,!?;:—\-])\1+', r'\1', line)
        line = re.sub(r'\.{3,}', '…', line)
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            cleaned.append(line)

    # Удаление дубликатов с сохранением порядка
    seen = set()
    unique_cleaned = []
    for line in cleaned:
        if line not in seen:
            seen.add(line)
            unique_cleaned.append(line)

    print(f"Очищено строк: {len(unique_cleaned)}")
    return unique_cleaned

cleaned_lines = load_and_preprocess("data")

Загрузка текстов...
🧹 Очистка и фильтрация...
Очищено строк: 295374


## 2. Чанкирование для подготовки к токенизации

In [ ]:
def chunk_texts(lines, chunk_char_size=1500):
    chunks = []
    current_chunk = []
    current_len = 0

    for line in lines:
        if current_len + len(line) + 1 > chunk_char_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = [line]
            current_len = len(line)
        else:
            current_chunk.append(line)
            current_len += len(line) + 1  # +1 for space

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    print(f"Создано чанков: {len(chunks)}")
    return chunks

chunks = chunk_texts(cleaned_lines)

Создано чанков: 28543


In [ ]:
chunks[:2]

['КАПИТАНСКАЯ ДОЧКА Береги честь смолоду. Пословица. СЕРЖАНТ ГВАРДИИ - Был бы гвардии он завтра ж капитан. - Того не надобно; пусть в армии послужит. - Изрядно сказано! пускай его потужит. . . . . . . . . . . . . . . . Да кто его отец? Княжнин. Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17. году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве.',
 'Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С пятилетнего возраста отдан я был на руки с

## 3. Обучение токенизатора

In [ ]:
def train_custom_tokenizer(text_chunks, vocab_size=3000, save_path="custom_tokenizer"):
    print("Обучение токенизатора...")
    tokenizer = Tokenizer(models.BPE())
    tokenizer.normalizer = None

    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
    tokenizer.decoder = decoders.ByteLevel()

    special_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"]
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=special_tokens)

    with tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".txt", encoding="utf-8") as f:
        f.write("\n".join(text_chunks))
        f.flush()
        tokenizer.train([f.name], trainer)
        os.remove(f.name)

    os.makedirs(save_path, exist_ok=True)
    tokenizer.save(os.path.join(save_path, "tokenizer.json"))

    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        bos_token="<bos>", eos_token="<eos>", pad_token="<pad>", unk_token="<unk>"
    )
    hf_tokenizer.save_pretrained(save_path)
    print(f"Токенизатор сохранён в {save_path}")
    return hf_tokenizer

tokenizer = train_custom_tokenizer(chunks, vocab_size=3000, save_path="custom_tokenizer")

Обучение токенизатора...



Токенизатор сохранён в custom_tokenizer


In [ ]:
test_text = "Все мысли, которые имеют огромные последствия, требуют внимания."
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens, skip_special_tokens=True)

print("Оригинал :", test_text)
print("Токены    :", tokens)
print("Декодирово:", decoded)
assert decoded.strip() == test_text.strip(), "Декодер работает некорректно!"

Оригинал : Все мысли, которые имеют огромные последствия, требуют внимания.
Токены    : [1036, 1752, 14, 1004, 463, 121, 807, 2275, 377, 1007, 2444, 14, 2117, 2569, 1444, 901, 16]
Декодирово:  Все мысли, которые имеют огромные последствия, требуют внимания.


## 4. Подготовка датасета

In [ ]:
def prepare_dataset(chunks, tokenizer, max_length=512):
    print("Токенизация и форматирование датасета...")
    dataset = Dataset.from_dict({"text": chunks})

    def tokenize_fn(examples):
        encodings = tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            add_special_tokens=True,
            padding=False
        )
        return encodings

    tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) >= 10)
    print(f"Итоговый размер датасета: {len(tokenized_ds)} примеров")
    return tokenized_ds

train_dataset = prepare_dataset(chunks, tokenizer, max_length=512)

Токенизация и форматирование датасета...


Map:   0%|          | 0/28543 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28543 [00:00<?, ? examples/s]

Итоговый размер датасета: 28536 примеров


In [ ]:
train_dataset[0]

{'input_ids': [271,
  1512,
  1469,
  2434,
  1859,
  1512,
  1167,
  1717,
  1726,
  1512,
  2381,
  298,
  1812,
  2540,
  1726,
  1512,
  392,
  158,
  360,
  123,
  167,
  1104,
  394,
  962,
  129,
  16,
  1380,
  805,
  2456,
  16,
  257,
  76,
  105,
  76,
  116,
  76,
  106,
  1512,
  1167,
  1859,
  382,
  1386,
  1512,
  76,
  116,
  1783,
  2434,
  2434,
  161,
  392,
  683,
  205,
  192,
  900,
  142,
  618,
  237,
  2126,
  216,
  1723,
  207,
  206,
  16,
  161,
  311,
  222,
  175,
  444,
  1447,
  29,
  136,
  2081,
  138,
  2864,
  123,
  525,
  248,
  207,
  16,
  161,
  1264,
  1896,
  165,
  322,
  1498,
  4,
  136,
  1035,
  336,
  268,
  432,
  248,
  207,
  16,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  558,
  908,
  268,
  1522,
  33,
  1879,
  236,
  2238,
  16,
  253,
  428,
  194,
  963,
  1704,
  2573,
  382,
  127,
  209,
  227,
  138,
  848,
  769,
  902,
  1411,
  166,
  293,
 

## 5. Инициализация модели

In [ ]:
import gc

# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [ ]:
print("Инициализация модели Llama...")
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=len(tokenizer),
    max_position_embeddings=512,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    use_cache=True
)

model = LlamaForCausalLM(config).to(device)
print(f"Количество параметров: {model.num_parameters():,}")

Инициализация модели Llama...
Количество параметров: 132,006,912


## 6. Коллбэк для валидации на промптах

In [ ]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

class GenerationEvalCallback(TrainerCallback):
    def __init__(self, prompts, tokenizer, device, log_every_steps=200):
        self.prompts = prompts
        self.tokenizer = tokenizer
        self.device = device
        self.log_every = log_every_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.log_every == 0:
            self._generate(state.global_step, kwargs["model"])

    def _generate(self, step, model):
        model.eval()
        print(f"\n{'='*20} EVAL STEP {step} {'='*20}")
        for prompt in self.prompts[:4]:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=40,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.1
                )
            gen_text = self.tokenizer.decode(out[0], skip_special_tokens=True)
            print(f"{prompt}\n {gen_text}\n")
        model.train()
        print(f"{'='*50}\n")

## 7. Настройка Trainer и запуск обучения

In [ ]:
training_args = TrainingArguments(
    output_dir="./rus_lit_llama_pretrain",
    per_device_train_batch_size=24,
    gradient_accumulation_steps=4,  # 24 * 4 = 96 эффективный batch_size
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    num_train_epochs=3,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="no",
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[GenerationEvalCallback(test_prompts, tokenizer, device, log_every_steps=200)]
)

print("Запуск обучения...")
trainer.train()

Запуск обучения...


Step,Training Loss
50,6.936187
100,6.406120
150,5.907480
200,5.549887
250,5.262088
300,5.064704
350,4.891594



==================== EVAL STEP 200 ====================
Все мысли, которые имеют огромные последствия
  Все мысли, которые имеют огромные последствия. Я с ними в тене и на него ее, что это, что он не только что это, что я не мог бы и, и я так же он не не не так#

Сила войска зависит от его духа
  Сила войска зависит от его духа в рерные и разая, что она, когда не села. Встя, и все выразится к нему. - Я, - и, не было не забыл,

Мысль о том, что он принес страдания
  Мысль о том, что он принес страдания. Вы не мог бы и не знаю, что он, когда я не только так как будто она не только и не с темом. Я бы, что я не любить. Я это не

Человек сознает себя свободным
  Человек сознает себя свободнымно не него, в бахром и мела на лале. Она калерил его на меня на Бавей. На что он в коле? А я к




Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 8. Финальная генерация на тестовых промптах

In [ ]:
model.eval()
model.to(device)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Промпт: {prompt}")
    print(f"Ответ:  {generated}")
    print("-"*60)

# SFT

In [41]:
import torch
from transformers import TrainerCallback
import numpy as np
import multiprocessing
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported

In [42]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET_NAME = "d0rj/alpaca-cleaned-ru"
OUTPUT_DIR = "./qwen2.5-0.5b-ru-sft-peft"
NUM_EPOCHS = 3
SEED = 42

DTYPE = None if is_bfloat16_supported() else torch.float16
LOAD_IN_4BIT = False

In [43]:
# Выбор числа workers для DataLoader
def get_optimal_num_workers():
    num_cpus = multiprocessing.cpu_count()
    workers = min(4, max(1, num_cpus - 1))
    print(f"CPU cores: {num_cpus}, using dataloader_num_workers: {workers}")
    return workers

NUM_WORKERS = get_optimal_num_workers()

CPU cores: 2, using dataloader_num_workers: 1


## 1. Загрузка токенизатора и модели

In [44]:
import gc

# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [45]:
print("Loading model and tokenizer...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=512,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

Loading model and tokenizer...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/Qwen2.5-0.5B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 2. Загрузка и подготовка датасета

In [46]:
# Фильтрация данных
def filter_valid(example):
    inst = example.get("instruction", "").strip()
    out = example.get("output", "").strip()
    return bool(inst and out)

In [47]:
def format_to_text(examples):
    texts = []
    instructions = examples["instruction"]
    inputs = examples.get("input", [""] * len(instructions))
    outputs = examples["output"]

    system_prompt = "Ты - ассисистент, отвечающий на запросы на русском языке."

    for inst, inp, out in zip(instructions, inputs, outputs):
        user_content = f"{inst}\n\nContext: {inp}" if inp.strip() else inst

        # Формат Qwen2.5: <|im_start|>role\ncontent<|im_end|>\n
        text = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{out.strip()}<|im_end|>"
        )
        texts.append(text)

    return {"text": texts}

In [48]:
# Загрузка и подготовка датасета
print(f"Loading dataset {DATASET_NAME}...")
raw_dataset = load_dataset(DATASET_NAME, split="train")
filtered_dataset = raw_dataset.filter(filter_valid, num_proc=4)
print(f"Filtered samples: {len(filtered_dataset)}")

formatted_dataset = filtered_dataset.map(
    format_to_text,
    remove_columns=filtered_dataset.column_names,
    batched=True,
    num_proc=NUM_WORKERS
)

split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Loading dataset d0rj/alpaca-cleaned-ru...
Filtered samples: 51760
Train: 49172, Eval: 2588


In [49]:
# Автоматический расчёт max_seq_length на основе датасета
def compute_max_seq_length(dataset, tokenizer, percentile=99.5, buffer=50, hard_limit=1024):
    field = "text" if "text" in dataset.column_names else "output"
    sample_size = min(5000, len(dataset))
    subset = dataset if len(dataset) <= sample_size else dataset.shuffle(seed=42).select(range(sample_size))

    lengths = []
    for example in subset:
        content = example.get(field, "")
        if content:
            length = len(tokenizer.encode(content, add_special_tokens=False))
            lengths.append(length)

    if not lengths:
        return 512

    max_length = int(np.percentile(lengths, percentile)) + buffer
    max_length = min(max_length, hard_limit)
    max_length = max(max_length, 256)

    print(f"Dataset length analysis: min={min(lengths)}, mean={np.mean(lengths):.0f}, "
          f"p99.5={np.percentile(lengths, 99.5):.0f}, selected max_seq_length={max_length}")

    return max_length

MAX_SEQ_LENGTH = compute_max_seq_length(train_dataset, tokenizer)

Dataset length analysis: min=54, mean=294, p99.5=925, selected max_seq_length=975


## 3. Обучение

In [56]:
test_prompts = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]

class TestPromptCallback(TrainerCallback):
    def __init__(self, model, tokenizer, test_prompts, system_prompt="Ты - ассистент, отвечающий на запросы на русском языке."):
        self.model = model
        self.tokenizer = tokenizer
        self.test_prompts = test_prompts
        self.system_prompt = system_prompt
        self.generation_kwargs = {
            "max_new_tokens": 256,
            "temperature": 0.7,
            "top_p": 0.9,
            "do_sample": True,
            "eos_token_id": tokenizer.eos_token_id,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id
        }

    def on_evaluate(self, args, state, control, logs, **kwargs):
        self.model.eval()
        device = self.model.device

        print(f"\n{'='*60}")
        print(f"[Step {state.global_step}] Инференс на тестовых промптах:")
        print(f"{'='*60}")

        with torch.no_grad():
            for prompt in self.test_prompts:
                chat_template = (
                    f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
                    f"<|im_start|>user\n{prompt}<|im_end|>\n"
                    f"<|im_start|>assistant\n"
                )
                inputs = self.tokenizer(chat_template, return_tensors="pt").to(device)
                outputs = self.model.generate(**inputs, **self.generation_kwargs)

                # Декодируем только сгенерированную часть
                input_len = inputs.input_ids.shape[1]
                response = self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
                print(f"Запрос: {prompt}")
                print(f"Ответ: {response}\n")

        print(f"{'='*60}\n")
        self.model.train()


eval_callback = TestPromptCallback(
    model=model,
    tokenizer=tokenizer,
    test_prompts=test_prompts
)

In [57]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    weight_decay=0.01,
    dataloader_num_workers=4,
    dataloader_prefetch_factor=NUM_WORKERS,
    dataloader_persistent_workers=True,
    report_to="none",
    seed=SEED,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [58]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    callbacks=[eval_callback]
)

Unsloth: You set `max_seq_length` as 975 but the maximum the model supports is 512. We shall reduce it.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# Обучение
print("Starting training...")
trainer.train()

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,172 | Num Epochs = 3 | Total steps = 4,611
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss,Validation Loss


In [ ]:
# Сохранение финальной модели
print(f"Saving model to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 4. Валидация

In [ ]:
def generate_response(prompt, model, tokenizer, max_new_tokens=256):
    messages = [
        {"role": "system", "content": "Ty — poleznyj assistent, otvechayushchij na voprosy na russkom yazyke."},
        {"role": "user", "content": prompt}
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant\n" in response:
        response = response.split("assistant\n")[-1].strip()
    return response

In [ ]:
for i, q in enumerate(test_prompts, 1):
    print(f"\nModel Input {i}:\n{q}")
    answer = generate_response(q, model, tokenizer)
    print(f"Model Output {i}:\n{answer}")
    print("-" * 40)